# Quickstart

**Combine a few language models into one "fusion," run it on real benchmark
questions, and measure whether the blend beats its best single model — in five
steps.**

This is the shortest path through screamingface: connect → pick → compose → run
→ compare. No scripts, no leaderboard, no tuning — just the core loop, end to
end. Want the fuller tour? Read [`00_overview`](00_overview.ipynb); after that
each step gets its own notebook.

**What you'll learn**
- how to connect a provider and pick a few diverse models
- how to compose them into a fusion whose recipe is a shareable `url4`
- how to run it on real questions and read the one number that matters: **gain**

> **Simulated but honest.** Benchmark *questions* are real; model *answers* are
> simulated deterministically, so this runs offline and gives the same result
> every time.

In [1]:
import warnings; warnings.simplefilter("ignore")
import screamingface as sf
sf.mock_widgets(True)   # render static widget previews (great on GitHub); delete this line to go interactive
sf.__version__

'0.2.0'

## 1 · Connect a provider

Models come from **providers** (Anthropic, OpenAI, …). `sf.setup()` opens a panel
with a **Connect** button and a masked key field for each. Your key resolves
from a Colab secret or an environment variable, or you type it once at a masked
prompt — either way it stays in memory, is never written to disk, and never
appears in a shared recipe.

*(Answers are simulated here, so connect real keys or skip ahead — the run works
either way.)* → deep dive: **[`01_authentication`](01_authentication.ipynb)**.

In [2]:
sf.setup()                       # connect Anthropic + OpenAI from the panel
# sf.connect("anthropic")        # …or from code: uses ANTHROPIC_API_KEY
# sf.connect("openai")           #    (secret → env var → masked prompt)

<widget preview — .value holds the object>

## 2 · Pick a few diverse models

`sf.models.list()` searches the catalog and hands back plain `provider/model`
ids. Filter by price (USD per million tokens) and take a few — the more
*different* the models, the more a fusion has to gain from combining them.

In [3]:
ids = sf.models.list(max_price=20)      # everything under $20 / M tokens
ids[:3]                                 # three to start with

['open_router/claude-sonnet-4.6',
 'open_router/gemini-2.5-pro',
 'open_router/gpt-4o']

## 3 · Compose a fusion

Give it a name, your models, and how to combine their answers. Here every member
votes (`majority_vote`) and one member — the **judge** — breaks ties. The judge
is just one of your models, named by its id. (Other reduce strategies:
`weighted_avg`, `best_of_n`, `merge`.)

Every fusion serialises to one `url4://` recipe. That string *is* the fusion's
identity — share it and anyone can rebuild the exact same setup. Keys are never
part of it.

In [4]:
fusion = sf.Fusion("fusion", models=ids[:3], reduce="majority_vote", judge=ids[0])
fusion.url                              # the shareable recipe

'url4://fusion?models=open_router/claude-sonnet-4.6+open_router/gemini-2.5-pro+open_router/gpt-4o&reduce=majority_vote&loop=parallel&judge=open_router/claude-sonnet-4.6'

## 4 · Run it on real questions

`fusion.evaluate()` runs the fusion on a benchmark. Here it's **GPQA Diamond** —
graduate-level science multiple-choice with a known answer key. `first=20` runs a
20-question sample; `seed` makes it reproducible — same seed, same result.

In [5]:
run = fusion.evaluate("gpqa", first=20, seed=0)
run

Run('fusion' on 'GPQA Diamond': score=50.0  gain=+0.0  cost=$0.123)

## 5 · The payoff — did the blend beat its best model?

Three numbers tell the story:

- **`score`** — the fusion's accuracy
- **`baseline`** — its best single member's accuracy
- **`gain`** — `score − baseline`

Read **`gain`** first. **Positive means the combination helped** — the models
covered each other's mistakes. A high score with zero gain just means one strong
model carried the group.

In [6]:
run.score, run.baseline, run.gain

(50.0, 50.0, 0.0)

## Recap

You ran the whole screamingface loop:

1. **Connect** a provider — `sf.setup()`
2. **Pick** diverse models — `sf.models.list(max_price=20)`
3. **Compose** a fusion — `sf.Fusion(…, reduce="majority_vote", judge=ids[0])`, shareable as `.url`
4. **Run** it on real questions — `fusion.evaluate("gpqa", first=20, seed=0)`
5. **Compare** — read the **`gain`**

That's the core. From here:
- [`02_models`](02_models.ipynb) → [`03_fusions`](03_fusions.ipynb) go deeper on
  each step (weights, prompts, cost, `stats`, the inspector).
- [`05_leaderboard`](05_leaderboard.ipynb) is where you prove it to **others** —
  publish your run and climb the board.